# Evaluation Script: HTTPS-By-Default Opt-Out

This script produces results for Section 5.2 HTTPS-By-Default Opt-Out

### Connect to the databases

In [1]:
import pymongo

client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["webview"]
manifest_info_collection = db["manifest_info"]
network_info_collection = db["network_info"]

def print_latex_macro(name: str, value: str):
    print(f"\\newcommand{{\\{name}}}{{{value}}}")

### Retrieve stats on the apps allowing all domains

In [2]:
all_apps_manifest_success = manifest_info_collection.distinct("package_name", {})
print_latex_macro("numAppsManifest", f"{len(all_apps_manifest_success):,}")

package_names_allowing_all = network_info_collection.distinct("package_name", {
    "allowed_domains": {"$in": ["*", "--$*", "--other*"] },
    "disallowed_domains": { "$in": [ [], None ] }
})

print_latex_macro("numAppsAllowingAllDomains", f"{len(package_names_allowing_all):,}")
print_latex_macro("numAppsAllowingAllDomainsPercent", f"{len(package_names_allowing_all) / len(all_apps_manifest_success) * 100:.2f}")

\newcommand{\numAppsManifest}{189,280}
\newcommand{\numAppsAllowingAllDomains}{63,868}
\newcommand{\numAppsAllowingAllDomainsPercent}{33.74}


### Retrieve stats on the apps allowing all at least one domain

In [3]:
package_names_with_allowed = network_info_collection.distinct("package_name", {
    "allowed_domains.0": { "$exists": True }
})
print_latex_macro("numAppsWithAllowedDomains", f"{len(package_names_with_allowed):,}")
print_latex_macro("numAppsWithAllowedDomainsPercent", f"{len(package_names_with_allowed) / len(all_apps_manifest_success) * 100:.2f}")

\newcommand{\numAppsWithAllowedDomains}{84,032}
\newcommand{\numAppsWithAllowedDomainsPercent}{44.40}


### Retrieve stats on the apps configuring any configuration

In [4]:
package_names_security_config = set(manifest_info_collection.distinct("package_name", {"has_network_security_config": True}))
package_names_uses_cleartext_traffic = set(manifest_info_collection.distinct("package_name", {"uses_cleartext_traffic": {"$ne": None}}))
total_package_names = set(manifest_info_collection.distinct("package_name"))

package_names_with_any_config = package_names_security_config.union(package_names_uses_cleartext_traffic)

print_latex_macro("appsDeclaringAnyConfigurationPercent", f"{len(package_names_with_any_config) / len(total_package_names) * 100:.2f}")

\newcommand{\appsDeclaringAnyConfigurationPercent}{50.30}


### Retrieve stats on the apps preventing all cleartext traffic

In [5]:
package_names_allowing_none = network_info_collection.distinct("package_name", {
    "allowed_domains": {"$in": [ [], None ] },
})

print_latex_macro("appsPreventCleartextTrafficToAllDomainsPercentage", f"{len(package_names_allowing_none) / len(all_apps_manifest_success) * 100:.2f}")

\newcommand{\appsPreventCleartextTrafficToAllDomainsPercentage}{55.60}


### Retrieve stats on most commonly used domains

In [6]:
print("\n\nTop 20 Allowed Domains:")
print("----------------")

pipeline = [
    {"$unwind": "$allowed_domains"},
    {
        "$group": {
            "_id": "$allowed_domains",
            "package_count": {"$addToSet": "$package_name"},
        }
    },
    {
        "$project": {
            "_id": 0,
            "domain": "$_id",
            "package_count": {"$size": "$package_count"},
        }
    },
    {"$sort": {"package_count": -1}},
]

results = list(network_info_collection.aggregate(pipeline))

for r in results[0:20]:
    domain = r["domain"]
    count = r['package_count']
    print(f"* {domain} ({count:,} apps)")



Top 20 Allowed Domains:
----------------
* * (64,969 apps)
* *.127.0.0.1 (19,148 apps)
* *.localhost (2,419 apps)
* *.amazon-adsystem.com (1,434 apps)
* *.10.0.2.2 (1,340 apps)
* *.android.bugly.qq.com (472 apps)
* *.ip-api.com (402 apps)
* *.google.com (324 apps)
* localhost (302 apps)
* *.cdn-creatives-tencent-prd.unityads.unitychina.cn (272 apps)
* *.cdn-store-icons-tencent-prd.unityads.unitychina.cn (272 apps)
* *.cdn-creatives-akamaistls-prd.unityads.unity3d.com (271 apps)
* *.cdn-creatives-akamaistls-prd.acquire.unity3dusercontent.com (271 apps)
* *.cdn-creatives-prd.unityads.unity3d.com (271 apps)
* *.cdn-creatives-geocdn-prd.unityads.unity3d.com (271 apps)
* *.cdn-creatives-akamai-prd.unityads.unity3d.com (271 apps)
* *.cdn-store-icons-highwinds-prd.unityads.unity3d.com (271 apps)
* *.cdn-store-icons-akamai-prd.unityads.unity3d.com (271 apps)
* *.cdn-creatives-highwinds-prd.unityads.unity3d.com (269 apps)
* *.cdn-creatives-akamaistls-re-prd.unityads.unity3d.com (269 apps)


### Retrieve stats on apps with configuration issues

In [7]:
apps_invalid_configs = network_info_collection.distinct("package_name", {
    "allowed_domains": {"$in": ["--$*", "--other*"] }
})

print_latex_macro("numAppsNSCInvalid", f"{len(apps_invalid_configs):,}")

\newcommand{\numAppsNSCInvalid}{42}


### Retrieve stats on target SDK

In [8]:
apps_lower_or_equal_27 = manifest_info_collection.distinct(
    "package_name", 
    {
        "$expr": {
            "$lte": [{"$toInt": "$target_sdk"}, 27]
        }
    }
)

print_latex_macro("appsTargetingSDKLETwentySevenPercentage", f"{len(apps_lower_or_equal_27)/len(all_apps_manifest_success)*100:.2f}")

\newcommand{\appsTargetingSDKLETwentySevenPercentage}{0.07}
